In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# =========================
# Paths
# =========================
BASE_DIR = Path.cwd()

DATA_PROCESSED = BASE_DIR / "data_processed"
FIG_DIR = BASE_DIR / "figures"

DATA_PROCESSED.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

csv_path = BASE_DIR / "historico_txref.csv"

# =========================
# Load B3 curve
# =========================
df = pd.read_csv(csv_path, sep=";")
df["data_referencia"] = pd.to_datetime(df["data_referencia"])
df["taxa"] = pd.to_numeric(df["taxa"], errors="coerce")
df["dias_corridos"] = pd.to_numeric(df["dias_corridos"], errors="coerce")

# filtrar convenção 252 e APR
df = df[df["convencao_dias"] == 252].copy()
df = df[df["tipo_taxa"] == "APR"].copy()

df = df.dropna(subset=["data_referencia", "dias_corridos", "taxa"])
df = df.sort_values(["data_referencia", "dias_corridos"]).reset_index(drop=True)

# =========================
# Helpers
# =========================
BASIS = 252

def annual_rate_to_discount_factor(rate_pct: np.ndarray, days: np.ndarray, basis: int = BASIS) -> np.ndarray:
    r = rate_pct / 100.0
    return 1.0 / ((1.0 + r) ** (days / basis))

def discount_factor_to_annual_rate(dfactor: np.ndarray, days: np.ndarray, basis: int = BASIS) -> np.ndarray:
    return (dfactor ** (-basis / days) - 1.0) * 100.0

def interpolate_df(target_days: np.ndarray, curve_days: np.ndarray, curve_rates_pct: np.ndarray, basis: int = BASIS) -> np.ndarray:
    curve_days = np.asarray(curve_days, dtype=float)
    curve_rates_pct = np.asarray(curve_rates_pct, dtype=float)

    tmp = (
        pd.DataFrame({"dias_corridos": curve_days, "taxa": curve_rates_pct})
        .drop_duplicates("dias_corridos")
        .sort_values("dias_corridos")
    )

    curve_days = tmp["dias_corridos"].to_numpy()
    curve_rates_pct = tmp["taxa"].to_numpy()

    dfs = annual_rate_to_discount_factor(curve_rates_pct, curve_days, basis=basis)
    target_dfs = np.interp(target_days, curve_days, dfs)

    return discount_factor_to_annual_rate(target_dfs, target_days, basis=basis)

def forward_rate(z1_pct: float, t1_days: int, z2_pct: float, t2_days: int, basis: int = BASIS) -> float:
    z1 = z1_pct / 100.0
    z2 = z2_pct / 100.0

    acc1 = (1.0 + z1) ** (t1_days / basis)
    acc2 = (1.0 + z2) ** (t2_days / basis)

    fwd = (acc2 / acc1) ** (basis / (t2_days - t1_days)) - 1.0
    return fwd * 100.0

def build_target_days_corridos(ref_date: pd.Timestamp) -> dict:
    d1 = ref_date + pd.DateOffset(years=1)
    d2 = ref_date + pd.DateOffset(years=2)
    d5 = ref_date + pd.DateOffset(years=5)
    d10 = ref_date + pd.DateOffset(years=10)

    return {
        "dc_1y": (d1 - ref_date).days,
        "dc_2y": (d2 - ref_date).days,
        "dc_5y": (d5 - ref_date).days,
        "dc_10y": (d10 - ref_date).days,
    }

# =========================
# Build daily forwards
# =========================
rows = []

for dt, grp in df.groupby("data_referencia"):
    curve_days = grp["dias_corridos"].to_numpy()
    curve_rates = grp["taxa"].to_numpy()

    if len(curve_days) < 5:
        continue

    targets = build_target_days_corridos(pd.Timestamp(dt))
    target_vertices_dc = np.array(
        [targets["dc_1y"], targets["dc_2y"], targets["dc_5y"], targets["dc_10y"]],
        dtype=float
    )


    if curve_days.min() > target_vertices_dc.min() or curve_days.max() < target_vertices_dc.max():
        continue

    try:
        interp_rates = interpolate_df(
            target_days=target_vertices_dc,
            curve_days=curve_days,
            curve_rates_pct=curve_rates,
            basis=BASIS
        )

        z_1y, z_2y, z_5y, z_10y = interp_rates
        dc_1y, dc_2y, dc_5y, dc_10y = target_vertices_dc.astype(int)

        fwd_1y1y = forward_rate(z_1y, dc_1y, z_2y, dc_2y, basis=BASIS)
        fwd_5y5y = forward_rate(z_5y, dc_5y, z_10y, dc_10y, basis=BASIS)

        spread = fwd_5y5y - fwd_1y1y

        rows.append({
            "date": dt,
            "dc_1y": dc_1y,
            "dc_2y": dc_2y,
            "dc_5y": dc_5y,
            "dc_10y": dc_10y,
            "z_1y": z_1y,
            "z_2y": z_2y,
            "z_5y": z_5y,
            "z_10y": z_10y,
            "fwd_1y1y": fwd_1y1y,
            "fwd_5y5y": fwd_5y5y,
            "spread_5y5y_minus_1y1y": spread,
        })

    except Exception as e:
        print(f"Erro em {dt}: {e}")
        continue

curve_df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
curve_df.to_csv(DATA_PROCESSED / "curve_forwards_252.csv", index=False)

print(curve_df.head())
print(curve_df.tail())
print(curve_df.shape)

        date  dc_1y  dc_2y  dc_5y  dc_10y       z_1y       z_2y       z_5y  \
0 2010-01-04    365    730   1826    3652  10.466540  11.804241  12.739961   
1 2010-01-05    365    730   1826    3652  10.443064  11.778530  12.742256   
2 2010-01-06    365    730   1826    3652  10.361769  11.741379  12.759936   
3 2010-01-07    365    730   1826    3652  10.358985  11.734230  12.789929   
4 2010-01-08    365    730   1826    3652  10.346181  11.727084  12.784542   

       z_10y   fwd_1y1y   fwd_5y5y  spread_5y5y_minus_1y1y  
0  13.139979  13.158140  13.541417                0.383277  
1  13.169972  13.130143  13.599310                0.469167  
2  13.199966  13.138236  13.641713                0.503477  
3  13.229962  13.126614  13.671712                0.545098  
4  13.209960  13.125269  13.636983                0.511714  
           date  dc_1y  dc_2y  dc_5y  dc_10y       z_1y       z_2y       z_5y  \
4004 2026-02-27    365    730   1826    3652  13.116803  12.609267  13.052967   
400